# Gout LLMOps — C01–C02

Mục tiêu: kiểm tra `chat_engine` bằng đúng luồng bước B. Notebook này **chưa làm history đa lượt và chưa làm RAG**. Khi chạy model thật, dùng revision đã khóa từ manifest B để giữ đúng trọng số.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
print('Uploaded:', zip_name)

In [ ]:
import zipfile, os, pathlib
work = pathlib.Path('/content/gout_c01_c02')
work.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(work)
root = work / 'gout_units_v01'
assert (root / 'unit03_chat_engine.py').exists(), 'ZIP thiếu unit03_chat_engine.py'
os.chdir(root)
print('Working dir:', os.getcwd())

In [ ]:
!python -m pip install -q -r requirements-unit02.txt

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python -m unittest discover -s tests -v

In [ ]:
import inspect
from unit03_chat_engine import ChatEngine
print(inspect.signature(ChatEngine.chat))
assert list(inspect.signature(ChatEngine.chat).parameters) == ['self','question','history','contexts']
print('C01 contract OK')

In [ ]:
from pathlib import Path
import shutil
out = Path('runs/c02_real')
if out.exists():
    shutil.rmtree(out)
!python unit03_chat_engine.py --config configs/unit03_hf_locked_from_b.json --out runs/c02_real

In [ ]:
import json
manifest = json.loads(Path('runs/c02_real/manifest.json').read_text(encoding='utf-8'))
pred = json.loads(Path('runs/c02_real/predictions.jsonl').read_text(encoding='utf-8'))
print('status:', pred['status'])
print('question:', pred['question'])
print('answer:', pred['answer'])
print('sources:', pred['sources'])
print('revision:', manifest.get('backend_info',{}).get('resolved_revision'))
print('finish_reason:', pred.get('call',{}).get('model_meta',{}).get('finish_reason'))
assert pred['status'] == 'ok'
assert pred['sources'] == []
assert manifest.get('reference_used') is False
assert manifest.get('rag_enabled') is False

In [ ]:
import shutil
archive = shutil.make_archive('/content/Gout_C02_real_result', 'zip', 'runs/c02_real')
print(archive)
files.download(archive)

Sau khi tải `Gout_C02_real_result.zip`, gửi lại file đó. Khi C02 thật `status=ok`, bước tiếp theo là **C03–C04**: tạo history rỗng cho case mới và append đúng câu trả lời thật của model sau lượt 1.